## 4.2 HARQ、AMC 与流控原理

在上一节中，我们了解了本章的学习目标和前置要求。本节介绍星闪 SLE MAC 层三项链路保障机制的核心原理。

本节学习大纲如下：

- HARQ Chase Combining 重传原理
- AMC 滑动窗口驱动的 MCS 调整
- QoS 流控背压机制

---

### 1. HARQ 重传

HARQ（Hybrid Automatic Repeat reQuest，混合自动重传）在 ARQ 基础上融合了信道编码的纠错能力。当前仿真采用 **Chase Combining（CC）** 简化模型：

- 发送端发送一帧，等待对端反馈 ACK/NAK
- 收到 ACK → 发送下一帧
- 收到 NAK → 重传相同的帧（最多重传 max_retries 次）
- 超过最大重传次数仍未成功 → 该帧丢弃，计入 HARQ FER

HARQ过程指的是包括传输数据、接收ACK/NACK以及必要时执行重传的传输和接收过程，如下图所示：

<img src="./images/HARQ.png" width="800">

Chase Combining 不改变码率，本质是用额外传输次数换取可靠性提升：

- 无 HARQ 吞吐量：$\text{TP}_{\text{no}} = (1 - \text{FER}) \times \text{SE}$
- HARQ 吞吐量：$\text{TP}_{\text{harq}} = (1 - \text{FER}_{\text{harq}}) \times \text{SE} / \text{AvgTx}$

其中 AvgTx 为每帧平均传输次数（≥1），反映了重传开销。

重传的核心逻辑如下：

```python
while not success and attempts <= max_retries:
    attempts += 1
    iq_retx = mac_to_iq(mac_payload, cfg)
    rx_iq_retx = _channel_impair(
        iq_retx, float(snr), channel_type, rician_k_db,
        cfo_hz, eq_method, cfg.sps, rng,
    )
    rx_retx = iq_to_mac(rx_iq_retx, cfg, n_mac_bytes)
    success = rx_retx.crc_ok and rx_retx.mac_payload == mac_payload
```

---

### 2. AMC 自适应 MCS 跟踪
#### 2.1 编码率
CodeRate = 有效比特数 / 编码后的比特数，即编码后的比特中包含有效信息的比率。如：

1/3编码，表示3个编码后的比特中，包含1个有效比特；

1/4编码，表示4个编码后的比特中，包含1个有效比特；

编码率越低，包含的冗余信息越多，纠错的能力越强，抗干扰的能力越强，传输的有效数据越小。

#### 2.2 编码效率

编码效率 = 编码率 * 调制阶数。

表示单个子载波能够承载的有效比特（不包括冗余信息）的位数。MCS 等级越高，编码效率越高。

#### 2.3 MCS

**MCS（Modulation and Coding Scheme，调制编码方案）** 是调制方式与码率的组合索引。本次仿真实验中定义了 13 个 MCS 等级（0–12）：
<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
    <tr>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:60px;">MCS</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:80px;">调制</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:100px;">编码率</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:100px;">编码效率</th>
        <th style="border:1px solid #ccc; padding:8px; text-align:left; min-width:100px;">调制阶数</th>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">BPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1/4</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0.250</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">BPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3/8</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0.375</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">QPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1/4</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0.500</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">QPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3/8</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">0.750</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">4</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">QPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1/2</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.000</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">5</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">QPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">5/8</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.250</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">6</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">QPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3/4</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.500</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">7</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">QPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">7/8</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.750</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">8</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">QPSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1/1</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2.000</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">9</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">8PSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">5/8</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1.875</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">10</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">8PSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3/4</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2.250</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">11</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">8PSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">7/8</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">2.625</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3</td>
    </tr>
    <tr>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">12</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">8PSK</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">1/1</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3.000</td>
        <td style="border:1px solid #ccc; padding:8px; text-align:left;">3</td>
    </tr>
</table>


MCS 越高编码效率越高但抗噪能力越弱——在差信道中需要用低 MCS 保证可靠性，在好信道中用高 MCS 提升速率。

#### 2.4 AMC

**AMC（Adaptive Modulation and Coding）** 就是根据信道质量动态选择最优 MCS 的机制。`LinkQualityTracker` 维护一个滑动窗口（默认 32 帧），记录每帧的 CRC 结果，计算窗口内 FER。

- FER < 1% 且 MCS 未达上限 -> 建议升 MCS（+1）
- FER > 10% 且 MCS 未达下限 -> 建议降 MCS（-1）
- 1% <= FER <= 10% -> 保持当前 MCS

核心决策函数 `suggest_mcs_adjustment()` 源码如下：

```python
def suggest_mcs_adjustment(self) -> int:
    if len(self._history) < self.window_size // 2:    # 窗口不满一半, 数据不足不调整
        return 0
    if self.fer < self.fer_target_low and self._current_mcs < 12:
        return 1                                       # FER < 1% 且未到上限 → 建议升 MCS
    if self.fer > self.fer_target_high and self._current_mcs > 0:
        return -1                                      # FER > 10% 且未到下限 → 建议降 MCS
    return 0                                           # 1% ≤ FER ≤ 10% → 保持当前
```

`apply_suggestion()` 将决策结果应用到内部状态，一次调用完成"决策 + 实施"：

```python
def apply_suggestion(self) -> int:
    adj = self.suggest_mcs_adjustment()      # 获取 ±1/0
    self.current_mcs = self._current_mcs + adj  # setter 自动钳位在 [0, 12]
    return self._current_mcs
```

滑动窗口大小影响跟踪性能：窗口太小 → 对随机波动敏感、容易过冲；窗口太大 → 反应迟钝。当 SNR 变化时，窗口内历史数据会造成滞后。具体源码可在后续 04.04 节中通过 `cat` 命令查看完整实现。

---

### 3. QoS 流控背压

QoS 流控机制通过限制发送队列长度和消费速率来防止发送方压垮接收方。`FlowController` 维护一个缓冲计数器 `buffer_count`，配合高/低水位线实现背压（backpressure）。

**核心数据结构**

```python
class FlowController:
    buffer_high_watermark: int = 16      # 高水位: 缓冲区上限
    buffer_low_watermark: int = 4        # 低水位: 缓冲区下限
    _buffer_count: int = 0               # 当前缓冲数据计数
    _paused: bool = False                # 是否已触发暂停
```

**enqueue / dequeue 逻辑**

```python
def enqueue(self, count: int = 1) -> None:
    self._buffer_count += count                      # 数据入队, 计数增加
    if self._buffer_count >= self.buffer_high_watermark:
        self._paused = True                          # 达到高水位 → 暂停接收

def dequeue(self, count: int = 1) -> None:
    self._buffer_count = max(0, self._buffer_count - count)  # 数据出队, 计数减少
    if self._paused and self._buffer_count <= self.buffer_low_watermark:
        self._paused = False                         # 降至低水位 → 恢复接收
```

**背压机制的工作流程**：

1. 发送端持续产生数据 → `enqueue()` 递增 `buffer_count`
2. `buffer_count ≥ high_watermark` → `_paused = True` → 发送端停止入队，数据暂存在上层
3. 接收端逐步消费数据 → `dequeue()` 递减 `buffer_count`
4. `buffer_count ≤ low_watermark` → `_paused = False` → 自动恢复接收

设置高低两个水位（而非单一门槛）的目的是避免频繁切换——如果只有一条线，缓冲区在临界值附近振荡会导致系统反复暂停/恢复，浪费资源。高低水位之间的间距起到类似施密特触发器的迟滞作用。

**流控的意义**：在射频链路中，物理层传输速率是有限的。当上层应用产生的数据超过物理层发送能力时，数据会在队列中堆积。如果不加限制，队列最终溢出导致丢包。流控通过背压将拥塞信号反向传递给发送端，使整个系统的数据注入速率自动匹配物理层的吞吐能力。

---
### 课后练习

1. HARQ Chase Combining 的核心思想是什么？
   - A. 每次重传使用不同的编码方式
   - B. 重传相同的帧，通过多次接收合并提升等效 SNR
   - C. 根据信道质量动态调整重传次数
   - D. 重传时降低调制阶数

2. LinkQualityTracker 中，MCS 调整的 FER 阈值分别是多少？
   - A. FER < 5% 升，FER > 20% 降
   - B. FER < 1% 升，FER > 10% 降
   - C. FER < 10% 升，FER > 50% 降
   - D. FER < 0.1% 升，FER > 1% 降

3. FlowController 为什么使用高低两个水位（而非单一门槛）？
   - A. 为了兼容不同带宽的信道
   - B. 避免缓冲区在临界值附近振荡导致反复暂停/恢复
   - C. 因为标准协议强制要求两个水位
   - D. 高低水位分别对应上行和下行流量

4. 滑动窗口大小对 AMC 跟踪性能的影响是什么？
   - A. 窗口越大跟踪越灵敏
   - B. 窗口太小易对随机波动过冲，窗口太大反应迟钝
   - C. 窗口大小不影响性能
   - D. 窗口大小只影响内存占用


执行以下代码获取答案


In [ ]:
!cat answer/04.02_answer.txt
